# Part A: Dataset Inspection

## 1. Dataset Dimensions and Overview
- **Number of Observations (Rows):** 4,630
- **Number of Features (Columns):** 9
- **Storage Profile:** All 9 fields are loaded initially as generic `object` (string) types, requiring deliberate type conversion and parsing for numerical, datetime, and categorical representations.

| Column Name | Initial Dtype | Intended Semantic Type | Description / Measurement Level |
| :--- | :--- | :--- | :--- |
| **`Company`** | `object` | Categorical (Nominal) | Organization or state entity conducting the mission. |
| **`Location`** | `object` | Categorical (Nominal) | Composite text string containing launch site, facility, and country. |
| **`Date`** | `object` | Datetime | Calendar launch date (`YYYY-MM-DD`). |
| **`Time`** | `object` | Datetime / Time | Launch timestamp in UTC (`HH:MM:SS`). |
| **`Rocket`** | `object` | Categorical (Nominal) | Specific launch vehicle system / model designation. |
| **`Mission`** | `object` | Categorical (Identifier) | Specific payload / mission project name. |
| **`RocketStatus`** | `object` | Categorical (Binary) | Vehicle operational status as of August 2022 (`Active` vs. `Retired`). |
| **`Price`** | `object` | Numerical (Continuous) | Reported rocket launch cost in millions of USD. |
| **`MissionStatus`** | `object` | Categorical (Target) | Final mission outcome (`Success`, `Failure`, `Partial Failure`, `Prelaunch Failure`). |

---

## 2. Missing Values and Duplicate Observations Audit
- **Duplicate Observations:** Exactly **1 duplicate row** was identified (Index 3530 and 3531: CASC / Long March 2D / 2008-11-05), which will be removed during cleaning to avoid artificial weight.
- **Missingness Profile:**
  - **`Price`:** Missing **3,365 values (72.68%)**. Imputing over 70% of a feature risks introducing severe noise and false precision; following standard preprocessing guidelines, dropping this column is recommended.
  - **`Time`:** Missing **127 values (2.74%)**.
  - **Other 7 Columns:** `Company`, `Location`, `Date`, `Rocket`, `Mission`, `RocketStatus`, and `MissionStatus` contain 0 missing values (100% complete).

---

## 3. High-Cardinality, Identifier, and Outlier Inspection
- **`Mission` Column:** Displays 4,556 unique names across 4,630 rows (98.4% unique). It serves as a unique transaction identifier rather than an informative predictive feature and will not generalize across modeling tasks.
- **`Rocket` Column:** Contains 370 distinct rocket families. While informative, many rockets appear only once or twice, requiring grouping or target encoding to manage cardinality.

---

## 4. Identification of Target Variable and Candidate Input Features
- **Possible Target Variable:** `MissionStatus`.
  - For supervised classification, it will be collapsed into a binary outcome: **`Success` (4,162 rows, 89.89%)** vs. **`Not-Success` (468 rows, 10.11%)**.
- **Viable Candidate Features ($X$):**
  - `Company` (organizational reliability and history).
  - `Country` (extracted from `Location` to represent launch geography and infrastructure).
  - `Launch_Year` (extracted from `Date` to capture technological era and operational maturity).
  - `Rocket` (vehicle family architecture).
- **Excluded Features:**
  - `Price` (discarded due to 72.7% missingness).
  - `Time` (lacks domain causal impact on structural flight success).
  - `Mission` (unique row identifier).
  - `RocketStatus` (excluded from predictive modeling to prevent temporal data leakage, as it reflects vehicle retirement status in 2022 rather than pre-flight conditions).

In [1]:
import numpy as np
import pandas as pd

# Load dataset while keeping raw reference intact
df_raw = pd.read_csv("space_missions.csv", encoding="latin1")
df = df_raw.copy()

# Part A: Dataset Inspection Outputs
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
print("\n--- Missing Values Count ---")
print(df.isna().sum()[df.isna().sum() > 0])
print(f"\nExact Duplicate Rows: {df.duplicated().sum()}")

# Target Variable Distribution
print("\n--- Target Variable Distribution (MissionStatus) ---")
status_dist = pd.DataFrame(
    {
        "Count": df["MissionStatus"].value_counts(),
        "Percentage": (df["MissionStatus"].value_counts(normalize=True) * 100)
        .round(2)
        .astype(str)
        + "%",
    }
)
print(status_dist)

Dataset Dimensions: 4630 rows, 9 columns

--- Missing Values Count ---
Time      127
Price    3365
dtype: int64

Exact Duplicate Rows: 1

--- Target Variable Distribution (MissionStatus) ---
                   Count Percentage
MissionStatus                      
Success             4162     89.89%
Failure              357      7.71%
Partial Failure      107      2.31%
Prelaunch Failure      4      0.09%


### Part B — Define the Possible Machine-Learning Problem

**Proposed Framework: Sequential Hybrid Strategy (Exploratory Clustering → Imbalanced Binary Classification)**

This project adopts a two-stage workflow: unsupervised clustering first as
exploratory pattern discovery, then supervised classification as the actual
predictive task. The clustering stage does not produce the final answer — it
informs how the classification stage should be set up.

---

#### Why not Regression?

The guide lists regression as an option, so it is worth stating explicitly why it
does not apply. Regression predicts a **continuous** numeric quantity. Our
outcome column, `MissionStatus`, is **discrete** — a launch either succeeded,
failed, partially failed, or failed before launch. There is no numeric value to
predict.

The only genuinely continuous candidate in the dataset was `Price`, and Part A
already eliminated it at 72.68% missing. So the dataset has no usable continuous
target, which rules regression out on two independent grounds.

---

#### Stage 1: Unsupervised Clustering (Exploratory Pattern Discovery)

* **Algorithm:** K-Means.
* **Objective:** Discover how space missions naturally group across time and
  operator history, without imposing outcome labels.
* **Why the Dataset Fits:** The dataset spans 65 years (1957–2022) of
  geopolitical shifts and evolving space infrastructure. Grouping missions by
  similarity across standardised numeric features (such as launch era and
  accumulated operator experience) should reveal latent operational archetypes,
  for example early experimental programmes versus mature modern commercial
  operations.
* **Why K-Means:** It groups records by similarity across numeric features and
  requires no prior knowledge of the groups. Two of its properties shape later
  decisions: it measures Euclidean distance, so the features must be
  **standardised** first (see Part D), and it requires the number of clusters
  *k* to be chosen in advance, so we compare several values of *k* rather than
  assuming one.
* **Role in Pipeline:** Clustering is an unsupervised diagnostic. Because it
  never sees the target, any agreement between cluster structure and failure rate
  is independent evidence that the structure is real rather than an artefact of
  the label. It also indicates which engineered features are worth building for
  prediction.

---

#### Stage 2: Supervised Classification (Pre-Launch Outcome Prediction)

* **Algorithm:** Decision Tree.
* **Objective:** Predict whether a planned mission will result in a
  **Success (1)** or **Not-Success / Failure (0)** prior to liftoff.
* **Target Framing:** Collapses `MissionStatus` into a binary label:
  `Success` (89.89%) vs. `Not-Success` (10.11%, encompassing Failure, Partial
  Failure, and Prelaunch Failure). Binarisation is necessary because
  `Prelaunch Failure` has only 4 observations, too few to learn or stratify.
* **Why a Decision Tree:** A tree splits the data with readable if/else rules on
  individual features. It fits this problem because:
  1. It handles **mixed feature types** — numeric and categorical — so not every
     feature must be scaled or one-hot encoded.
  2. It is **interpretable**: each root-to-leaf path reads as a plain rule, such
     as "fewer than 20 prior launches by this operator AND launch year before
     1975 → elevated failure risk." That kind of explanation is what a risk
     question demands.
  3. It captures **non-linear relationships and interactions**, such as failure
     risk depending on era *and* operator maturity together rather than on either
     alone.
  4. It is **robust to irrelevant features**, since it simply will not split on a
     column that adds no separation.
* **How Clustering Informs Classification:**
  1. *Feature Derivation:* Any structure found in Stage 1 justifies engineering
     maturity features, such as accumulated operator experience, rather than
     assuming they matter.
  2. *Evaluation Awareness:* If failures cluster in earlier, less mature
     operational environments, then a naive random split risks an unrepresentative
     test set — reinforcing the need for stratification.
* **Handling Class Imbalance:** Because the target is roughly 90/10, raw accuracy
  is rejected as a performance metric. A model that predicts `Success` for every
  mission would score about 90% accuracy while detecting zero failures. The
  classification stage will therefore use stratified train/test splits and be
  evaluated with minority-class Precision, Recall, F1, and PR-AUC.

---

#### Candidate Target and Features

* **Target (`y`):** `MissionStatus`, binarised to `Success` vs `Not-Success`.
* **Candidate Features (`X`):** `Company`, `Country` (extracted from `Location`),
  `Launch_Year` (extracted from `Date`), `Rocket`, and derived operator-experience
  features.
* **Excluded:** `Price` (72.7% missing), `Mission` (98.4% unique identifier),
  `Time` (no causal bearing on vehicle reliability), and `RocketStatus`
  (temporal leakage — see Part F).